# 한 프로세스 안의 두 할당자 — GPU 메모리 풀 합치기[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SangwookBaek/SangwookBaek.github.io/blob/main/assets/notebooks/gpu-memory-pool-sharing.ipynb)**묻는 것.** 한 프로세스에서 PyTorch 와 CuPy 를 같이 쓸 때, **torch 가 놓아준 메모리를 cupy 가 쓸 수 있는가.**답은 기본값에서 "못 쓴다"다. 둘 다 자기 캐시를 따로 쥐기 때문이다. 그래서 카드에 빈 자리가있는데도 OOM 이 난다. 이 노트북은 그 상황을 재현하고, [RMM](https://github.com/rapidsai/rmm) 으로풀을 하나로 합쳐 해결한 뒤, **합치면 무엇을 잃는지**까지 측정한다.## 측정 방법라이브러리 카운터가 아니라 **드라이버가 보는 여유 메모리**(`cudaMemGetInfo`)를 본다.풀을 합치면 `torch.cuda.memory_allocated()` 가 예외를 던지기 때문에(STEP 4), 두 회차에서똑같이 읽을 수 있는 값이 이것뿐이다. `nvidia-smi` 가 보여주는 것과 같은 숫자다.## 두 번 돌린다`torch.cuda.memory.change_current_allocator()` 는 **첫 CUDA 할당 이전에만** 호출할 수 있다.따라서 한 세션에서 두 모드를 볼 수 없다.| 회차 | `USE_RMM` | 하는 일 ||---|---|---|| 1 | `False` | 전체 실행 → 결과가 `results_rmm_off.json` 에 저장된다 || 2 | `True` | **런타임 재시작**(Colab: `런타임 > 세션 다시 시작`) → 전체 실행 |두 회차가 끝나면 맨 아래 판정표가 자동으로 채워진다.> 코랩에서 **GPU 런타임**인지 확인할 것. `런타임 > 런타임 유형 변경 > T4 GPU`.

## 준비

In [ ]:
!pip install -q rmm-cu12 --extra-index-url=https://pypi.nvidia.com

### 이번 회차 설정`USE_RMM` 만 바꿔가며 두 번 돌린다.

In [ ]:
USE_RMM = False          # 1회차 False → 런타임 재시작 → 2회차 TrueSIZE_MIB = 512           # 한 덩어리 크기RETAIN_MIB = 1024        # 풀이 캐시로 쥐고 있을 크기 (release threshold)# STEP 8 은 GPU 를 거의 꽉 채운다. 세션이 죽으면 재시작하고 False 로 두면 된다.RUN_FILL_TEST = TrueHEADROOM_MIB = 256       # 채운 뒤 남겨둘 여유. SIZE_MIB 보다 작아야 실험이 성립한다RESULTS_PATH = f"results_rmm_{'on' if USE_RMM else 'off'}.json"print(f"USE_RMM={USE_RMM}  SIZE_MIB={SIZE_MIB}  RETAIN_MIB={RETAIN_MIB}  →  {RESULTS_PATH}")

### 헬퍼

In [ ]:
import gcimport jsonMIB = 1024 * 1024RESULTS = {"use_rmm": USE_RMM, "size_mib": SIZE_MIB, "retain_mib": RETAIN_MIB}def free_mib():    # 드라이버가 보는 여유 메모리. nvidia-smi 와 같은 관점이다    import torch    free, _total = torch.cuda.mem_get_info()    return free // MIBdef record(key, value):    RESULTS[key] = value    print(f"  {key:32} = {value}")def floats(mib):    return mib * MIB // 4def raw_cuda_malloc(mib):    # 어떤 풀도 거치지 않는 생 cudaMalloc. 성공하면 포인터, 실패하면 None    import cupy as cp    try:        return cp.cuda.runtime.malloc(mib * MIB)    except Exception:        return Nonedef raw_cuda_free(ptr):    import cupy as cp    if ptr is not None:        cp.cuda.runtime.free(ptr)def try_alloc(fn):    # 할당을 시도하고 (성공여부, 객체) 를 준다. 실패 사유는 라이브러리마다 다르다    try:        return True, fn()    except Exception as exc:        print(f"    실패: {type(exc).__name__}")        return False, None

## STEP 0 — 환경여기서는 **CUDA 를 건드리지 않는다.** 임포트와 버전 출력만 한다.`change_current_allocator()` 가 STEP 1 에서 성공해야 하므로, 그 앞에 CUDA 할당을 만드는호출을 두지 않는다.

In [ ]:
import cupy as cpimport rmmimport torchprint(f"torch {torch.__version__}")print(f"cupy  {cp.__version__}")print(f"rmm   {rmm.__version__}")print()!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## STEP 1 — 풀 꽂기**이 셀이 첫 CUDA 할당이어야 한다.**`USE_RMM=False` 면 아무것도 하지 않는다. 그게 베이스라인이다 — torch 와 cupy 가 각자 자기캐시를 쓴다.`USE_RMM=True` 면 세 가지를 한다.1. `CudaAsyncMemoryResource(release_threshold=...)` — `cudaMallocAsync` 기반 풀을 만든다.   `release_threshold` 는 **드라이버에 반납하지 않고 쥐고 있을 크기**다.2. torch 의 할당자를 이 풀로 교체한다.3. cupy 의 할당자도 이 풀로 교체한다.`StatisticsResourceAdaptor` 는 풀을 감싸 장부를 보여주는 관측용 계층이다. 실제로 얼마가풀을 통해 나갔는지 STEP 3 에서 이걸로 교차 확인한다.

In [ ]:
pool_installed = Falsestats = Noneif USE_RMM:    from rmm.allocators.cupy import rmm_cupy_allocator    from rmm.allocators.torch import rmm_torch_allocator    pool = rmm.mr.CudaAsyncMemoryResource(release_threshold=RETAIN_MIB * MIB)    stats = rmm.mr.StatisticsResourceAdaptor(pool)    rmm.mr.set_current_device_resource(stats)    torch.cuda.memory.change_current_allocator(rmm_torch_allocator)    cp.cuda.set_allocator(rmm_cupy_allocator)    pool_installed = True    print(f"공유 풀 설치 완료 — release_threshold={RETAIN_MIB} MiB")    print("  torch → rmm")    print("  cupy  → rmm")else:    print("베이스라인 — torch 와 cupy 가 각자 자기 풀을 쓴다")record("pool_installed", pool_installed)record("driver_free_at_start_mib", free_mib())

## STEP 2 — `allocated` 와 `reserved` (1회차 전용)풀을 합치기 전에, 애초에 왜 메모리가 안 빠지는지부터 본다. PyTorch 는 두 개의 카운터를 준다.```pythontorch.cuda.memory_allocated()   # 텐서에게 실제로 넘어간 양torch.cuda.memory_reserved()    # 드라이버한테 받아온 총량```텐서를 놓으면 `allocated` 는 줄지만 **`reserved` 는 그대로다.** `cudaMalloc`/`cudaFree` 가느리기 때문에 PyTorch 가 의도적으로 쥐고 있는 것이다. 이게 캐싱 할당자다.2회차에서는 이 카운터가 예외를 던져서 건너뛴다 — 그 자체가 STEP 4 의 내용이다.

In [ ]:
if pool_installed:    print("2회차에서는 건너뛴다 — torch 카운터를 읽을 수 없다 (STEP 4 참고)")else:    t = torch.empty(floats(SIZE_MIB), dtype=torch.float32, device="cuda")    print("텐서 생성 후")    print(f"  allocated {torch.cuda.memory_allocated() // MIB:5} MiB")    print(f"  reserved  {torch.cuda.memory_reserved() // MIB:5} MiB")    print(f"  드라이버 여유 {free_mib():5} MiB")    del t    gc.collect()    print("\n텐서 해제 후 — allocated 만 줄었다")    print(f"  allocated {torch.cuda.memory_allocated() // MIB:5} MiB")    print(f"  reserved  {torch.cuda.memory_reserved() // MIB:5} MiB")    print(f"  드라이버 여유 {free_mib():5} MiB")    before = free_mib()    torch.cuda.empty_cache()    print("\nempty_cache() 후 — 이제야 드라이버로 돌아간다")    print(f"  reserved  {torch.cuda.memory_reserved() // MIB:5} MiB")    print(f"  드라이버 여유 {free_mib():5} MiB  (+{free_mib() - before})")

## STEP 3 — 본론: torch 가 놓은 자리를 cupy 가 쓰는가이 노트북의 핵심 셀이다. 순서는 이렇다.1. torch 로 `SIZE_MIB` 를 잡는다 → 드라이버 여유가 그만큼 줄어든다2. 그 텐서를 놓는다 → **드라이버 여유는 돌아오지 않는다** (캐시가 쥐고 있다)3. cupy 로 같은 크기를 잡는다 → **여기서 갈린다**`cupy_took_mib` 가 0 에 가까우면 cupy 가 torch 가 놓은 자리를 재사용한 것이다.`SIZE_MIB` 에 가까우면 드라이버한테 새로 받아온 것이다.

In [ ]:
base = free_mib()t = torch.empty(floats(SIZE_MIB), dtype=torch.float32, device="cuda")after_torch = free_mib()del tgc.collect()after_free = free_mib()x = cp.empty(floats(SIZE_MIB), dtype=cp.float32)after_cupy = free_mib()record("torch_took_mib", base - after_torch)record("returned_on_free_mib", after_free - after_torch)record("cupy_took_mib", after_free - after_cupy)print()if RESULTS["cupy_took_mib"] < SIZE_MIB // 4:    print(f"→ cupy 가 torch 의 빈 자리를 재사용했다 ({RESULTS['cupy_took_mib']} MiB 만 추가로 씀)")else:    print(f"→ cupy 가 드라이버한테 새로 받아왔다 ({RESULTS['cupy_took_mib']} MiB). 두 캐시는 서로를 못 본다")del xgc.collect()

### 풀 장부로 교차 확인2회차에서만 의미가 있다. 위에서 torch 와 cupy 가 요청한 양이 **같은 장부**에 찍혔는지 본다.`total_count` 가 두 라이브러리의 할당을 합친 값이면 실제로 한 풀을 쓰는 것이다.

In [ ]:
if stats is None:    print("베이스라인 — 공유 장부가 없다")else:    s = stats.allocation_counts    print(s)    record("pool_total_count", s.total_count)    record("pool_peak_mib", s.peak_bytes // MIB)

## STEP 4 — 잃는 것 ①: torch 카운터풀을 갈아끼우면 PyTorch 는 자기가 할당한 게 아니므로 장부를 쓸 수 없다.`memory_allocated()` / `memory_reserved()` 가 예외를 던진다.**이게 중요한 이유.** 이 카운터를 읽는 모니터링이나 로깅이 이미 있다면, 풀을 합치는 순간그쪽이 깨진다. 메모리 문제를 진단하려고 도입한 도구가 진단 수단을 없애는 셈이다.

In [ ]:
try:    a = torch.cuda.memory_allocated()    r = torch.cuda.memory_reserved()    print(f"    allocated={a // MIB} MiB   reserved={r // MIB} MiB")    record("torch_counters", "읽힘")except Exception as exc:    record("torch_counters", f"{type(exc).__name__}")    print(f"    {exc}")

## STEP 5 — 잃는 것 ②: cupy 기본 풀이 no-op 이 된다cupy 의 할당이 RMM 으로 가면, cupy 의 **기본 메모리 풀**은 아무것도 담지 않는다.따라서 `free_all_blocks()` 는 예외 없이 조용히 아무 일도 하지 않는다.**이게 중요한 이유.** 메모리를 비우려고 `free_all_blocks()` 를 부르던 코드가 있으면, 그 줄은살아 있지만 효과만 사라진다. 에러가 안 나기 때문에 눈치채기도 어렵다.

In [ ]:
default_pool = cp.get_default_memory_pool()before_used = default_pool.used_bytes()y = cp.empty(floats(SIZE_MIB), dtype=cp.float32)record("default_pool_backs_alloc", default_pool.used_bytes() > before_used)del ygc.collect()before_free = free_mib()default_pool.free_all_blocks()record("free_all_blocks_returned_mib", free_mib() - before_free)

## STEP 6 — `release_threshold` 는 상한이 아니다가장 오해하기 쉬운 부분이다. `release_threshold` 는 **태스크 사이에 캐시로 쥐고 있을 크기**이고할당 상한이 아니다. 그보다 큰 걸 요청하면 그냥 할당된다.`RETAIN_MIB` 의 두 배를 잡아본다.- **peak** 이 2 × `RETAIN_MIB` 로 올라가면 → 상한이 아니라는 증거- 그 뒤 풀이 쥐고 있는 양이 `RETAIN_MIB` 로 줄면 → 임계값이 캐시 크기로 동작한다는 증거이 셀은 STEP 6 시작 시점이 아니라 **풀이 비어 있던 STEP 1 시점**을 기준으로 잰다.여기 올 때쯤이면 앞 단계들이 남긴 캐시가 조금 쥐여 있어서, 증분으로 재면 그만큼 어긋난다.그런데 **해제만으로는 안 돌아온다.** `cudaMallocAsync` 는 임계값을 **동기화 지점에서** 적용한다.그래서 해제 후 한 번 더, `torch.cuda.synchronize()` 뒤에 측정한다.실용적으로 중요한 함의가 있다. 커널을 안 돌리고 대기만 하는 유휴 프로세스는 동기화할 일이없으므로 **임계값을 넘겨 쥔 메모리를 계속 붙들고 있는다.** 같은 카드를 쓰는 다른 프로세스입장에서는 그게 그냥 없는 메모리다.베이스라인에서는 torch 가 전부 쥐고 있어 `empty_cache()` 를 부를 때까지 안 돌아온다.

In [ ]:
base = free_mib()big = torch.empty(floats(2 * RETAIN_MIB), dtype=torch.float32, device="cuda")peak = free_mib()del biggc.collect()settled = free_mib()torch.cuda.synchronize()synced = free_mib()# STEP 6 에 들어올 때 이미 캐시가 조금 쥐여 있으므로, 증분이 아니라# 풀이 비어 있던 STEP 1 시점을 기준으로 잰다.zero = RESULTS["driver_free_at_start_mib"]record("peak_taken_mib", base - peak)record("pool_holds_after_free_mib", zero - settled)record("pool_holds_after_sync_mib", zero - synced)print()print(f"→ 임계값 {RETAIN_MIB} MiB 인데 {RESULTS['peak_taken_mib']} MiB 를 잡았다 (상한이 아니다)")print(f"→ 해제 직후 쥐고 있는 양 {RESULTS['pool_holds_after_free_mib']} MiB — 해제만으로는 안 돌아온다")print(f"→ synchronize() 뒤 {RESULTS['pool_holds_after_sync_mib']} MiB")before = free_mib()torch.cuda.empty_cache()record("empty_cache_returned_mib", free_mib() - before)if pool_installed and RESULTS["empty_cache_returned_mib"] == 0:    print("→ empty_cache() 는 no-op 이다. RMM 메모리에 손이 닿지 않는다")

## STEP 7 — 풀 밖 할당자는 풀의 여유를 못 본다풀을 합쳐도 해결되지 않는 게 있다. **자기 스스로 `cudaMalloc` 을 부르는 라이브러리**다.CUDA 솔버나 직접 만든 커널이 흔히 그렇게 한다.풀이 `RETAIN_MIB` 를 캐시로 쥐고 있는 상태에서 생 `cudaMalloc` 을 부른다. 풀의 여유를재사용한다면 드라이버 여유가 그대로여야 한다.

In [ ]:
warm = torch.empty(floats(SIZE_MIB), dtype=torch.float32, device="cuda")del warmgc.collect()before = free_mib()ptr = raw_cuda_malloc(SIZE_MIB)after = free_mib()record("raw_malloc_took_mib", before - after)raw_cuda_free(ptr)print()print("→ 풀이 같은 크기를 비워둔 채로 있었는데도 드라이버에서 새로 받아갔다")print("  풀 밖 할당자를 편입하려면 그 라이브러리가 제공하는 할당자 훅에 풀을 물려야 한다")

## STEP 8 — 카드를 채우면 (하이라이트)지금까지는 여유가 넉넉해서 낭비가 눈에 안 보였다. 실제 OOM 은 카드가 찼을 때 난다.절차는 이렇다.1. `HEADROOM_MIB` 만 남기고 torch 텐서로 GPU 를 채운다2. `SIZE_MIB` 짜리 하나를 **놓는다** → 캐시가 그걸 쥐고 있고, 드라이버 여유는 `HEADROOM_MIB` 뿐이다3. 같은 크기를 세 가지 방법으로 요청한다| 요청 | 1회차 (풀 따로) | 2회차 (풀 공유) ||---|---|---|| cupy | **실패** — torch 캐시를 못 본다 | **성공** — 같은 풀에서 받는다 || 생 `cudaMalloc` | 실패 | 실패 — 합쳐도 풀 밖은 못 본다 || torch | 성공 (대조군) | 성공 (대조군) |`HEADROOM_MIB` 가 `SIZE_MIB` 보다 작아야 실험이 성립한다. 넉넉하면 드라이버에서 그냥 받아버린다.

In [ ]:
if not RUN_FILL_TEST:    print("RUN_FILL_TEST=False — 건너뛴다")else:    hold = []    while free_mib() - SIZE_MIB > HEADROOM_MIB:        ok, block = try_alloc(            lambda: torch.empty(floats(SIZE_MIB), dtype=torch.float32, device="cuda")        )        if not ok:            break        hold.append(block)    record("filled_blocks", len(hold))    record("driver_free_when_full_mib", free_mib())    victim = hold.pop()    del victim    gc.collect()    record("driver_free_after_release_mib", free_mib())    print(f"\n{SIZE_MIB} MiB 를 놓았지만 드라이버 여유는 그대로다 — 캐시가 쥐고 있다\n")    print(f"cupy 로 {SIZE_MIB} MiB 요청")    ok, obj = try_alloc(lambda: cp.empty(floats(SIZE_MIB), dtype=cp.float32))    record("cupy_alloc_when_full", "성공" if ok else "실패")    del obj    gc.collect()    print(f"\n생 cudaMalloc 으로 {SIZE_MIB} MiB 요청")    ptr = raw_cuda_malloc(SIZE_MIB)    record("raw_alloc_when_full", "성공" if ptr is not None else "실패")    raw_cuda_free(ptr)    print(f"\ntorch 로 {SIZE_MIB} MiB 요청 (대조군)")    ok, obj = try_alloc(        lambda: torch.empty(floats(SIZE_MIB), dtype=torch.float32, device="cuda")    )    record("torch_alloc_when_full", "성공" if ok else "실패")    del obj, hold    gc.collect()

## STEP 9 — 결과 저장

In [ ]:
with open(RESULTS_PATH, "w") as f:    json.dump(RESULTS, f, indent=2, ensure_ascii=False)print(f"{RESULTS_PATH} 저장")print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 판정표두 회차를 모두 돌리면 아래가 채워진다. 1회차만 돌린 상태면 `USE_RMM=True` 로 바꾸고**런타임을 재시작한 뒤** 처음부터 다시 실행한다.

In [ ]:
import osROWS = [    ("cupy_took_mib", "STEP 3 — cupy 가 추가로 쓴 양", f"≈{SIZE_MIB} (공유 안 됨)", "≈0 (공유됨)"),    ("torch_counters", "STEP 4 — torch 카운터", "읽힘", "예외"),    ("default_pool_backs_alloc", "STEP 5 — cupy 기본 풀이 할당을 담는가", "True", "False"),    ("free_all_blocks_returned_mib", "STEP 5 — free_all_blocks() 회수량", "> 0", "0"),    ("peak_taken_mib", "STEP 6 — 임계값의 2배 요청 시 peak", f"≈{2 * RETAIN_MIB}", f"≈{2 * RETAIN_MIB}"),    ("pool_holds_after_free_mib", "STEP 6 — 해제 직후 쥐고 있는 양", "쥐고 있음", "쥐고 있음"),    ("pool_holds_after_sync_mib", "STEP 6 — synchronize() 뒤 쥐고 있는 양", "그대로", f"≈{RETAIN_MIB}"),    ("empty_cache_returned_mib", "STEP 6 — empty_cache() 회수량", "> 0", "0"),    ("raw_malloc_took_mib", "STEP 7 — 생 cudaMalloc 이 드라이버에서 가져간 양", f"≈{SIZE_MIB}", f"≈{SIZE_MIB}"),    ("cupy_alloc_when_full", "STEP 8 — 꽉 찬 상태에서 cupy", "실패", "성공"),    ("raw_alloc_when_full", "STEP 8 — 꽉 찬 상태에서 생 cudaMalloc", "실패", "실패"),    ("torch_alloc_when_full", "STEP 8 — 꽉 찬 상태에서 torch", "성공", "성공"),]def load(path):    if not os.path.exists(path):        return None    with open(path) as f:        return json.load(f)off, on = load("results_rmm_off.json"), load("results_rmm_on.json")if off is None or on is None:    have = "off" if off else ("on" if on else "없음")    print(f"두 회차가 필요하다. 지금 가진 것: {have}")else:    w = max(len(label) for _, label, _, _ in ROWS)    header = f"{'항목':<{w}} | {'기대 off':>16} | {'실측 off':>16} | {'기대 on':>16} | {'실측 on':>16}"    print(header)    print("-" * len(header))    for key, label, exp_off, exp_on in ROWS:        a = off.get(key, "—")        b = on.get(key, "—")        print(f"{label:<{w}} | {exp_off:>16} | {str(a):>16} | {exp_on:>16} | {str(b):>16}")

---## 정리**문제.** 한 프로세스 안의 두 할당자는 서로의 캐시를 못 본다. torch 가 15 GB 를 쥐고 그중14 GB 가 비어 있어도, cupy 입장에서 그건 존재하지 않는 메모리다. 카드에 자리가 있는데 OOM 이나는 이유다.**해결.** 둘을 같은 풀에 물리면 한쪽이 놓은 자리를 다른 쪽이 쓴다 (STEP 3, STEP 8).**대가.** 공짜가 아니다.- torch 의 메모리 카운터가 예외를 던진다 (STEP 4) — 진단 수단이 사라진다- cupy 의 `free_all_blocks()` 가 조용히 no-op 이 된다 (STEP 5) — 에러가 안 나서 더 위험하다- 반납이 임계값에 묶이고, 그마저 **동기화 지점에서만** 일어난다 (STEP 6) — 유휴 프로세스는  초과분까지 계속 쥐고 있다. 같은 카드를 쓰는 다른 프로세스에게는 그게 없는 메모리다**한계.** 풀을 합쳐도 **스스로 `cudaMalloc` 을 부르는 라이브러리**는 여전히 굶는다 (STEP 7, STEP 8).그 라이브러리가 할당자 훅을 제공해야 편입할 수 있고, 안 하는 라이브러리가 하나라도 있으면풀 통합만으로는 문제가 남는다.**따라서** 도입 판단은 "공유가 되는가"가 아니라 **"내 프로세스의 할당자를 전부 편입할 수 있는가"** 다.하나라도 밖에 남으면 통합 이득은 부분만 받고 관측성과 반납은 잃는다.## 이 노트북이 다루지 않은 것- **파편화** — 여기 할당 패턴은 크기가 다 같다. 실제 OOM 은 크기가 뒤섞여 생긴 구멍에서 난다- **여러 스트림 / 멀티 스레드** — 풀은 스트림 단위 동작이 따로 있다- **실제 워크로드** — 모델의 텐서 크기 분포는 이보다 훨씬 불규칙하다